# SOXX/SOXL Spread Oscillation Notebook

目的：按 notebook-first 规范复核 `SOXX_SPREAD_OSCILLATION`。假设来自 SOXL/SOXX 价差扩大、ratio z-score、spread volatility 和 relative choppiness；结论只决定是否进入后续训练/门控，不直接修改训练 profile。

目标门槛沿用 factor promotion gate：`abs(IC mean) > 0.03`、`IC+ > 0.55`、`profit factor > 1.2`、`signal accuracy > 0.55`、`max core correlation <= 0.80`。

## External Basis

- [Qlib GitHub](https://github.com/microsoft/qlib): 继续使用 qlib 风格的可复现实验边界。
- [Qlib arXiv 2009.11189](https://arxiv.org/abs/2009.11189): 将数据、模型、回测、分析放在同一 workflow 中验证。
- [Avellaneda & Zhang, Path-Dependence of Leveraged ETF Returns](https://papers.ssrn.com/sol3/papers.cfm?abstract_id=1404708): LETF 与 underlying 的偏离和 realized variance/path dependence 有机制基础。
- [Cheng & Madhavan, The Dynamics of Leveraged and Inverse ETFs](https://papers.ssrn.com/sol3/papers.cfm?abstract_id=1393995): LETF 日内再杠杆和微观结构影响提醒我们不能把价差信号直接当稳定 alpha。
- [Probability of Backtest Overfitting](https://www.davidhbailey.com/dhbpapers/backtest-overfitting.pdf): 多阈值/多候选筛选后必须用 gate 和 walk-forward 纪律防止过拟合。

In [1]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from scripts.evaluate_factor_pack import (
    _load_ohlcv_frames,
    evaluate_pack,
)
from src.soxl_quant.app.factor_diagnostics import (
    build_factor_gate_summary,
    render_factor_gate_summary,
)
from src.soxl_quant.data import engineer_feature_frame

ROOT = Path.cwd()
CARD_DIR = ROOT / 'runtime/factor/card/soxx_spread_oscillation'
PACK = 'soxx_spread_oscillation'
FEATURES = (
    'SOXL_SOXX_RATIO_Z_252',
    'SOXL_SOXX_RATIO_WIDEN_5',
    'SOXL_SOXX_SPREAD_VOL_Z_21',
    'SOXL_SOXX_CHOP_DIVERGENCE_21',
)
HORIZONS = (1, 5, 10, 20)
THRESHOLDS = (-1.0, -0.5, 0.0, 0.5, 1.0, 1.5)
external_references = [
    {'label': 'Qlib', 'url': 'https://github.com/microsoft/qlib', 'note': 'workflow and reproducibility boundary'},
    {'label': 'Qlib arXiv', 'url': 'https://arxiv.org/abs/2009.11189', 'note': 'AI-oriented quantitative investment workflow'},
    {'label': 'Avellaneda Zhang LETF path dependence', 'url': 'https://papers.ssrn.com/sol3/papers.cfm?abstract_id=1404708', 'note': 'LETF path dependence and realized variance basis'},
    {'label': 'Cheng Madhavan LETF dynamics', 'url': 'https://papers.ssrn.com/sol3/papers.cfm?abstract_id=1393995', 'note': 'daily rebalancing and microstructure basis'},
    {'label': 'Backtest overfitting', 'url': 'https://www.davidhbailey.com/dhbpapers/backtest-overfitting.pdf', 'note': 'multiple-testing guard for factor screens'},
]


In [2]:
ic_df, backtest_df, corr_df = evaluate_pack(
    pack_name=PACK,
    feature_names=FEATURES,
    output_dir=ROOT / 'runtime/factor',
    horizons=HORIZONS,
    thresholds=THRESHOLDS,
    write=False,
)
summary = build_factor_gate_summary(ic_df, backtest_df, corr_df)

ic_df.to_csv(CARD_DIR / 'ic_soxx_spread_oscillation_notebook.csv', index=False)
backtest_df.to_csv(CARD_DIR / 'backtest_soxx_spread_oscillation_notebook.csv', index=False)
corr_df.to_csv(CARD_DIR / 'correlation_soxx_spread_oscillation_notebook.csv', index=False)
summary.to_csv(CARD_DIR / 'gate_summary_soxx_spread_oscillation.csv', index=False)

display(summary)


,feature_name,best_ic_horizon,best_ic_mean,best_abs_ic_mean,best_ic_t,best_ic_positive_ratio,best_backtest_direction,best_threshold,best_sharpe,best_profit_factor,best_signal_accuracy,best_max_drawdown,max_abs_core_correlation,max_abs_core_correlation_feature,ic_gate,backtest_gate,correlation_gate,full_gate,primary_failure
0,SOXL_SOXX_SPREAD_VOL_Z_21,20,0.015606,0.015606,2.602862,0.488108,long,-1.0,0.846404,1.178295,0.543520,-0.911949,0.917273,vol_coef_21,False,False,False,False,correlation_redundant
1,SOXL_SOXX_RATIO_Z_252,20,-0.004930,0.004930,-0.796480,0.550740,long,-1.0,0.778425,1.160731,0.543271,-0.801773,0.884521,anchor_gap_180,False,False,False,False,correlation_redundant
2,SOXL_SOXX_RATIO_WIDEN_5,5,-0.002802,0.002802,-2.523733,0.491182,long,-1.0,0.774733,1.144643,0.534752,-0.933676,0.922091,mom_5,False,False,False,False,correlation_redundant
3,SOXL_SOXX_CHOP_DIVERGENCE_21,20,-0.001599,0.001599,-0.240868,0.425211,long,-1.0,0.827882,1.156962,0.536890,-0.905055,0.081352,tracking_error_21,False,False,True,False,weak_direction_and_execution


In [3]:
soxl, soxx = _load_ohlcv_frames()
feature_df, _ = engineer_feature_frame(soxl, soxx, feature_packs=(PACK,))
flat = feature_df.droplevel('instrument').copy()
returns = flat['return'].astype(float)

regime = pd.Series('base', index=flat.index, name='regime')
regime_risk = flat.get('regime_risk', pd.Series(0.5, index=flat.index)).astype(float)
mom_21 = flat.get('mom_21', pd.Series(0.0, index=flat.index)).astype(float)
regime.loc[regime_risk >= 0.65] = 'stress'
regime.loc[(regime_risk < 0.45) & (mom_21 > 0.0)] = 'bull'

slice_rows = []
for row in summary.itertuples(index=False):
    signal = flat[row.feature_name].astype(float)
    if row.best_backtest_direction == 'long':
        active = signal > float(row.best_threshold)
        next_correct = returns.shift(-1) > 0.0
    else:
        active = signal < float(row.best_threshold)
        next_correct = returns.shift(-1) < 0.0
    for regime_name in ['bull', 'base', 'stress']:
        mask = regime == regime_name
        active_mask = active & mask
        slice_rows.append({
            'feature_name': row.feature_name,
            'regime': regime_name,
            'rows': int(mask.sum()),
            'active_days': int(active_mask.sum()),
            'active_ratio': float(active_mask.sum() / max(mask.sum(), 1)),
            'next_return_mean_active': float(returns.shift(-1)[active_mask].mean()) if active_mask.any() else 0.0,
            'signal_accuracy_active': float(next_correct[active_mask].mean()) if active_mask.any() else 0.0,
        })
regime_slices = pd.DataFrame(slice_rows)
regime_slices.to_csv(CARD_DIR / 'regime_slice_soxx_spread_oscillation.csv', index=False)
display(regime_slices)


,feature_name,regime,rows,active_days,active_ratio,next_return_mean_active,signal_accuracy_active
0,SOXL_SOXX_SPREAD_VOL_Z_21,bull,1726,1177,0.681924,0.002991,0.539507
1,SOXL_SOXX_SPREAD_VOL_Z_21,base,1354,1201,0.887001,0.004232,0.551207
2,SOXL_SOXX_SPREAD_VOL_Z_21,stress,724,724,1.000000,0.003097,0.537293
3,SOXL_SOXX_RATIO_Z_252,bull,1726,1670,0.967555,0.002078,0.529341
4,SOXL_SOXX_RATIO_Z_252,base,1354,1063,0.785081,0.004165,0.558796
5,SOXL_SOXX_RATIO_Z_252,stress,724,306,0.422652,0.000828,0.565359
6,SOXL_SOXX_RATIO_WIDEN_5,bull,1726,1726,1.000000,0.002032,0.528389
7,SOXL_SOXX_RATIO_WIDEN_5,base,1354,1336,0.986706,0.003894,0.545659
8,SOXL_SOXX_RATIO_WIDEN_5,stress,724,650,0.897790,0.001928,0.529231
9,SOXL_SOXX_CHOP_DIVERGENCE_21,bull,1726,1724,0.998841,0.002048,0.528422


In [4]:
review = render_factor_gate_summary(summary, external_references=external_references)
(CARD_DIR / 'gate_summary_soxx_spread_oscillation.md').write_text(review, encoding='utf-8')

full_pass_count = int(summary['full_gate'].sum()) if not summary.empty else 0
correlation_failures = summary[summary['primary_failure'] == 'correlation_redundant']['feature_name'].tolist()
weak_failures = summary[summary['primary_failure'].str.contains('weak', na=False)]['feature_name'].tolist()
orthogonal_chop = summary[summary['feature_name'] == 'SOXL_SOXX_CHOP_DIVERGENCE_21'].iloc[0].to_dict()

conclusion_md = f'''## Runtime Conclusion

Decision: `blocked_no_feature_promotion`. This pack should not be used for direct model training now.

Evidence:

- Full-pass feature count: `{full_pass_count}` out of `{len(summary)}`.
- Correlation-redundant features: `{', '.join(correlation_failures) or 'none'}`. These mainly re-express existing trend/momentum/volatility dimensions.
- Weak direction/execution features: `{', '.join(weak_failures) or 'none'}`.
- `SOXL_SOXX_CHOP_DIVERGENCE_21` remains the only orthogonal spread/oscillation candidate (`max rho={orthogonal_chop['max_abs_core_correlation']:.3f}`), but its best IC is `{orthogonal_chop['best_ic_mean']:+.4f}` with IC+ `{orthogonal_chop['best_ic_positive_ratio']:.3f}`, so it lacks directional stability.

Implication for the active overfit objective:

- Do not launch a SOXX/SOXL spread feature retrain from this evidence. It would likely add redundant inputs and increase overfit risk.
- If this family is revisited, use it only as a regime-gated execution diagnostic or as one term inside a walk-forward-weighted multi-objective selector, not as a standalone core feature.
- The next factor loop should prefer orthogonal candidates from the gate-gap context (`vol_of_vol_21`, `tracking_error_vol_21`, `decay_63`) or an explicitly new external mechanism with IC and walk-forward validation.
'''
(CARD_DIR / 'runtime_conclusion_ipynb.md').write_text(conclusion_md, encoding='utf-8')

selection = {
    'experiment': 'soxx_spread_oscillation_ipynb_v1',
    'decision': 'blocked_no_feature_promotion',
    'pack': PACK,
    'features': list(FEATURES),
    'full_pass_count': full_pass_count,
    'correlation_redundant_features': correlation_failures,
    'weak_features': weak_failures,
    'orthogonal_candidate': 'SOXL_SOXX_CHOP_DIVERGENCE_21',
    'orthogonal_candidate_ic_mean': float(orthogonal_chop['best_ic_mean']),
    'orthogonal_candidate_ic_positive_ratio': float(orthogonal_chop['best_ic_positive_ratio']),
    'orthogonal_candidate_max_core_correlation': float(orthogonal_chop['max_abs_core_correlation']),
    'next_action': 'do_not_retrain_directly; only revisit as regime-gated diagnostic or selector term',
    'artifacts': {
        'notebook': str((CARD_DIR / 'soxx_spread_oscillation.ipynb').relative_to(ROOT)),
        'ic': str((CARD_DIR / 'ic_soxx_spread_oscillation_notebook.csv').relative_to(ROOT)),
        'backtest': str((CARD_DIR / 'backtest_soxx_spread_oscillation_notebook.csv').relative_to(ROOT)),
        'correlation': str((CARD_DIR / 'correlation_soxx_spread_oscillation_notebook.csv').relative_to(ROOT)),
        'gate_summary': str((CARD_DIR / 'gate_summary_soxx_spread_oscillation.csv').relative_to(ROOT)),
        'regime_slice': str((CARD_DIR / 'regime_slice_soxx_spread_oscillation.csv').relative_to(ROOT)),
        'review': str((CARD_DIR / 'gate_summary_soxx_spread_oscillation.md').relative_to(ROOT)),
        'runtime_conclusion': str((CARD_DIR / 'runtime_conclusion_ipynb.md').relative_to(ROOT)),
    },
    'external_references': external_references,
}
(CARD_DIR / 'selection_ipynb.json').write_text(json.dumps(selection, ensure_ascii=False, indent=2), encoding='utf-8')

display(Markdown(conclusion_md))


## Runtime Conclusion

Decision: `blocked_no_feature_promotion`. This pack should not be used for direct model training now.

Evidence:

- Full-pass feature count: `0` out of `4`.
- Correlation-redundant features: `SOXL_SOXX_SPREAD_VOL_Z_21, SOXL_SOXX_RATIO_Z_252, SOXL_SOXX_RATIO_WIDEN_5`. These mainly re-express existing trend/momentum/volatility dimensions.
- Weak direction/execution features: `SOXL_SOXX_CHOP_DIVERGENCE_21`.
- `SOXL_SOXX_CHOP_DIVERGENCE_21` remains the only orthogonal spread/oscillation candidate (`max rho=0.081`), but its best IC is `-0.0016` with IC+ `0.425`, so it lacks directional stability.

Implication for the active overfit objective:

- Do not launch a SOXX/SOXL spread feature retrain from this evidence. It would likely add redundant inputs and increase overfit risk.
- If this family is revisited, use it only as a regime-gated execution diagnostic or as one term inside a walk-forward-weighted multi-objective selector, not as a standalone core feature.
- The next factor loop should prefer orthogonal candidates from the gate-gap context (`vol_of_vol_21`, `tracking_error_vol_21`, `decay_63`) or an explicitly new external mechanism with IC and walk-forward validation.
